# Aula 02 — Interpretando mensagens com um modelo aberto e gratuito

**UC9 — Compreender e Aplicar Machine Learning em soluções de IA**  
**Projeto integrador:** Agentes de IA para o setor imobiliário  
**Duração:** 3 horas presenciais

## O que construiremos

Nesta aula, criaremos a primeira parte de um agente imobiliário capaz de:

1. receber uma mensagem escrita normalmente por um cliente;
2. usar um modelo de linguagem aberto para interpretar a mensagem;
3. transformar o texto em dados estruturados;
4. validar os dados com Pydantic;
5. usar regras Python para escolher o próximo passo.

### Exemplo de entrada

> Quero comprar um apartamento de dois quartos em Águas Claras. Posso pagar até R$ 450 mil e talvez precise financiar.

### Resultado esperado

```json
{
  "finalidade": "compra",
  "tipo_imovel": "apartamento",
  "regiao_desejada": "Águas Claras",
  "orcamento_maximo": 450000,
  "quantidade_quartos": 2,
  "precisa_financiamento": true
}
```

> **Não utilizaremos OpenAI API, cartão, créditos ou chave paga.**  
> O modelo será baixado e executado dentro do Google Colab.


## Objetivos de aprendizagem

Ao final da aula, você deverá conseguir:

- explicar a diferença entre modelo de linguagem e agente;
- carregar um modelo aberto com a biblioteca Transformers;
- enviar mensagens nos papéis `system` e `user`;
- gerar uma resposta localmente, sem API comercial;
- orientar o modelo a devolver JSON;
- extrair o JSON da resposta;
- validar o resultado com Pydantic;
- separar a interpretação feita pela IA das regras determinísticas em Python;
- testar o protótipo com mensagens diferentes.


## Fluxo resumido em seis passos:

1) Carrega o tokenizer → responsável por converter texto em tokens e vice-versa.

2) Carrega o modelo → baixa os pesos do LLM e o prepara para uso.
3) Formata a conversa → organiza as mensagens no formato esperado pelo modelo (chat template).
4) Tokeniza a entrada → transforma o texto em tensores numéricos e os envia para CPU ou GPU.
5) Gera novos tokens → o modelo prevê, um a um, os próximos tokens mais prováveis até atingir o limite ou encontrar o fim da resposta.
6) Decodifica os tokens → converte os novos tokens novamente em texto legível e retorna a resposta ao usuário.

# 1. Preparação do Google Colab

Antes de executar o notebook:

1. Abra **Ambiente de execução**.
2. Escolha **Alterar tipo de ambiente de execução**.
3. Em acelerador de hardware, escolha uma GPU disponível, preferencialmente **T4**.
4. Salve a configuração.

O notebook também possui uma alternativa menor para execução sem GPU, mas ela será mais lenta e poderá interpretar as mensagens com menor qualidade.


In [2]:
# Instalação das bibliotecas utilizadas na aula.


!pip install -U "transformers>=4.45.0" "accelerate>=0.34.0" "pydantic>=2.8.0"


# 2. Verificando o ambiente

A célula seguinte identifica se existe uma GPU disponível.

Usaremos:

- **Qwen2.5 1.5B Instruct** quando houver GPU;
- **Qwen2.5 0.5B Instruct** como alternativa mais leve quando houver apenas CPU.

Os dois modelos são públicos e não exigem chave de API.


In [3]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("GPU disponível?", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Aviso: o modelo será executado pela CPU e poderá ficar mais lento.")


Python: 3.12.13
PyTorch: 2.11.0+cu128
GPU disponível? True
GPU: Tesla T4


In [4]:
# Escolha automática do modelo conforme o hardware disponível.

MODELO_GPU = "Qwen/Qwen2.5-1.5B-Instruct"
MODELO_CPU = "Qwen/Qwen2.5-0.5B-Instruct"

MODELO_ID = MODELO_GPU if torch.cuda.is_available() else MODELO_CPU

print("Modelo selecionado:", MODELO_ID)


Modelo selecionado: Qwen/Qwen2.5-1.5B-Instruct


# 3. Carregando o modelo aberto

Na primeira execução, os arquivos do modelo serão baixados para o ambiente do Colab.

Isso pode levar alguns minutos. Depois de carregado, o texto será processado no próprio ambiente, sem cobrança por mensagem.

### Conceitos importantes

- **Tokenizer:** transforma texto em números que o modelo entende.
- **Modelo:** recebe os tokens e prevê os próximos tokens.
- **Pesos:** parâmetros aprendidos durante o treinamento.
- **Instruct:** versão ajustada para seguir instruções e conversar.


In [5]:
#AutoTokenizer -> responsável por transformar Tokens em texto (e vice-e-versa)
#AutoModelForCausalLM -> é o próprio modelo de linguagem (ele que recebe os tokens e prevê o próximo)
from transformers import AutoModelForCausalLM, AutoTokenizer


In [ ]:
# Carrega o tokenizer de acordo com o modelo escolhido. OBS: Cada modelo tem o seu.
tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)


In [ ]:
# Carregando o mnodelo:
#Baixa o modelo do Hugging Face e carrega oos bilhões de parâmetros da memórica
model = AutoModelForCausalLM.from_pretrained(
    MODELO_ID,
    torch_dtype="auto", #Deixa o Pytorch escolher os diferentes tipos de numeros. Usa menos memória,
    device_map="auto", #Pergunta automaticamente se existe GPU
    low_cpu_mem_usage=True #Tenta economizar RAM
    )

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
# Coloca no modo de inferência. Deixa o processo mais eficiente
model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [ ]:
#Pergunta onde o modelo está rodando e armazena o local
dispositivo_modelo = next(model.parameters()).device
print("Dispositivo do modelo:", dispositivo_modelo)

Dispositivo do modelo: cuda:0


# 4. Criando uma função para conversar com o modelo

Modelos de chat esperam mensagens com papéis:

- `system`: regras e comportamento;
- `user`: solicitação do usuário;
- `assistant`: respostas anteriores, quando houver histórico.

A função abaixo aplica o formato de conversa correto do modelo e gera somente a continuação produzida pelo assistente.


In [ ]:
#Formato padrão de mensagens
mensagens = [
    {
        "role": "user",
        "content":"Quem descobriu o Brasil?",
    }
]

In [ ]:
#outro formato:
<user>
Quem descobriu o Brasil?
<assistant>


In [ ]:
def gerar_resposta(
    mensagens: list[dict[str, str]],
    max_novos_tokens: int = 300,
) -> str:
    """Gera uma resposta usando o modelo aberto carregado no notebook."""

    #Transformando a mensagem no formato esperado:
    texto_formatado = tokenizer.apply_chat_template( # o apply_chat_template monta o formato automaticamente
        mensagens,
        tokenize=False,
        add_generation_prompt=True, #Coloca no final da converso o "Assistant:"
    )
    #print(texto_formatado)

    #Tokenizando -> Transformar texto em números
    entradas = tokenizer(
        texto_formatado,
        return_tensors="pt" #Retorna tensores do PyTorch
    )
    #print(entradas)

    #Envia cada item da entrada para onde o modelo está rodando
    entradas = {
        nome: tensor.to(dispositivo_modelo)
        for nome, tensor in entradas.items()
    }

    #Entrando no modo inferência do pytorch:
    with torch.inference_mode():
      #É a parte mais importante, é onde a IA começa aa gerar a resposta, prevendo um token por vez
      saidas = model.generate(
          **entradas, #Os ** significa: desempacotar o dict
          max_new_tokens=max_novos_tokens, #Limita a quantidade máxima de tokens que poderão ser gerados. Evita resposta grandes ou infinitas
          do_sample=False, # Sempre escolhe a palavra mais provável. O True, deixa ele mais criativo
          repetition_penalty=1.05, #Prevenir repeticoes de palavras.
          pad_token_id=tokenizer.eos_token_id, #Define qual é o token que será utilizado para preencher espaços quando necessário.
      )

    quantidade_tokens_entrada = entradas["input_ids"].shape[1] #Conta quantos tokens existiam originalmente.
    novos_tokens = saidas[0][quantidade_tokens_entrada:] #Aqui serve para descartarmos a pergunta. Ficamos apenas com as resposta.

    #Converter para texto:
    return tokenizer.decode(novos_tokens, skip_special_tokens=True).strip() # Remove tokens especiais, elimina espaços em branco no inicio o no final da resposta.




## Primeiro teste

Neste momento, ainda não estamos construindo um agente completo. Estamos verificando se o modelo consegue receber e interpretar uma mensagem em português.


In [ ]:
mensagens_teste = [
    {
        "role": "system",
        "content": (
            "Você é um assistente imobiliário. "
            "Responda em português, de forma curta e objetiva."
        ),
    },
    {
        "role": "user",
        "content": (
            "Quero comprar um apartamento de dois quartos em Águas Claras, "
            "até R$ 450 mil. O que você entendeu?"
        ),
    },
]




In [ ]:
resposta_teste = gerar_resposta(mensagens_teste, max_novos_tokens=300)

In [ ]:
resposta_teste

'Entendi que você está procurando por um apartamento com duas camas em Águas Claras, com valor máximo de R$ 450 mil.'

# 5. Definindo a estrutura esperada com Pydantic

Não queremos que o restante do sistema dependa de uma resposta em texto livre.

O modelo de linguagem ficará responsável por **interpretar a linguagem humana**.  
O Pydantic ficará responsável por **validar a estrutura e os tipos dos dados**.

Por exemplo:

- orçamento deve ser número ou `null`;
- quantidade de quartos deve ser número inteiro ou `null`;
- finalidade deve ter apenas valores previamente permitidos;
- campos extras não devem ser aceitos.


In [ ]:
from typing import Literal  #O Literal permite limitar um campo a valores específicos.


## Testando o Pydantic antes de usar a IA

É importante compreender a validação separadamente.


In [ ]:
dados_validos = {
    "finalidade": "compra",
    "tipo_imovel": "apartamento",
    "regiao_desejada": "Águas Claras",
    "orcamento_maximo": 450000,
    "quantidade_quartos": 2,
    "precisa_financiamento": True,
    "assunto_sensivel": False,
    "resumo": "Cliente procura apartamento de 2 quartos em Águas Claras."
}



Durante esse processo, o Pydantic verifica:

- presença dos campos obrigatórios;
- tipos dos dados;
- valores aceitos pelo Literal;
- campos adicionais;
- tamanho do resumo;
- validações personalizadas.

Caso tudo esteja certo, é criado um objeto:

Esse objeto [cliente_validado] não é mais apenas um dicionário. Ele é uma instância da classe QualificacaoCliente.

Poderíamos acessar seus atributos: print(cliente_validado.finalidade)

In [ ]:
# Exemplo propositalmente inválido.

dados_invalidos = {
    "finalidade": "troca",
    "tipo_imovel": "apartamento",
    "regiao_desejada": "Águas Claras",
    "orcamento_maximo": -100,
    "quantidade_quartos": 0,
    "precisa_financiamento": "talvez",
    "assunto_sensivel": False,
    "resumo": "Teste"
}



# 6. Orientando o modelo a produzir JSON

O modelo deve receber uma instrução muito clara:

- devolver somente JSON;
- usar exatamente os nomes dos campos;
- não inventar dados;
- usar `null` quando a informação estiver ausente;
- converter valores monetários para número;
- usar valores padronizados para a finalidade.

Mesmo assim, o modelo pode errar. Por isso, ainda precisaremos extrair e validar o JSON.


In [ ]:
INSTRUCAO_EXTRACAO = '''
Você é um componente de extração de informações de uma imobiliária.

Analise a mensagem do cliente e responda SOMENTE com um objeto JSON válido.
Não use Markdown, não use ``` e não escreva explicações fora do JSON.

Use exatamente estas chaves:
{
  "finalidade": "compra | locacao | nao_informado",
  "tipo_imovel": "texto ou null",
  "regiao_desejada": "texto ou null",
  "orcamento_maximo": "número ou null",
  "quantidade_quartos": "inteiro ou null",
  "precisa_financiamento": "true, false ou null",
  "assunto_sensivel": "true ou false",
  "resumo": "resumo curto em português"
}

Regras:
1. Não invente informações.
2. Use null quando a informação não estiver presente.
3. Converta expressões como "450 mil" para 450000.
4. Use "locacao", sem cedilha ou acento, no campo finalidade.
5. Marque assunto_sensivel como true quando o cliente pedir orientação
   jurídica, contratual ou uma análise financeira detalhada.
6. Dúvida simples sobre necessidade de financiamento não é, sozinha,
   uma orientação financeira detalhada.
'''


# 7. Extraindo o objeto JSON da resposta

Embora a instrução peça somente JSON, um modelo pequeno pode acrescentar algum texto.

A função seguinte procura o primeiro objeto JSON válido dentro da resposta.


# 8. Validando a interpretação

Agora temos três camadas distintas:

```text
Mensagem do cliente
        ↓
Modelo aberto interpreta
        ↓
Função extrai o JSON
        ↓
Pydantic valida os dados
```

Se alguma camada falhar, o sistema deve mostrar um erro controlado em vez de continuar com dados incorretos.


# 9. Criando uma função completa de interpretação

Esta função reúne:

1. montagem das mensagens;
2. geração local;
3. extração do JSON;
4. validação com Pydantic;
5. tratamento dos erros mais comuns.


# 10. O modelo interpreta; o Python decide

Não devemos entregar todas as decisões ao modelo de linguagem.

Uma arquitetura mais segura é:

- **modelo aberto:** interpreta a mensagem;
- **Pydantic:** valida os dados;
- **Python:** aplica as regras da imobiliária;
- **humano:** assume situações sensíveis ou de maior responsabilidade.

Nesta aula, o Python escolherá entre três ações:

1. solicitar uma informação ausente;
2. consultar imóveis;
3. encaminhar para um corretor humano.


# 11. Testando mensagens diferentes

Avalie se o modelo:

- extrai corretamente o tipo de imóvel;
- interpreta valores como “meio milhão”;
- usa `null` quando faltam dados;
- diferencia compra de locação;
- detecta assuntos que exigem intervenção humana.

> Modelos pequenos podem errar. O objetivo não é esconder os erros, mas aprender a identificá-los e controlá-los.


# 12. Registrando os resultados dos testes

Agentes não devem ser avaliados apenas pela impressão de que “parecem funcionar”.

Crie casos de teste e compare:

- resultado esperado;
- resultado produzido;
- erro encontrado;
- possível ajuste na instrução ou nas regras.


# Atividade prática — Parte 1

Em dupla ou trio:

1. Execute o notebook até a função `interpretar_mensagem_cliente`.
2. Crie pelo menos cinco mensagens de clientes.
3. Inclua formas diferentes de informar o orçamento:
   - `450 mil`;
   - `R$ 450.000`;
   - `meio milhão`;
   - `até uns 500k`.
4. Registre os erros encontrados.
5. Ajuste a instrução para tentar reduzir os erros.


In [ ]:
# Escreva aqui os cenários criados pelo grupo.

cenarios_do_grupo = [
    "",
    "",
    "",
    "",
    "",
]

for cenario in cenarios_do_grupo:
    if not cenario.strip():
        continue

    cliente, resposta_modelo = interpretar_mensagem_cliente(cenario)

    print("\nMensagem:", cenario)
    print("Resposta bruta:", resposta_modelo)

    if cliente:
        print("Validado:", cliente.model_dump())
        print("Ação:", decidir_proxima_acao(cliente))


# Atividade prática — Parte 2

Modifique o protótipo para extrair mais duas informações relevantes para o projeto imobiliário.

Sugestões:

- necessidade de vaga de garagem;
- prazo para mudança;
- preferência por imóvel mobiliado;
- área mínima;
- possui animal de estimação;
- aceita contato por WhatsApp;
- possui imóvel para vender;
- urgência do atendimento.

Você deverá:

1. alterar a classe Pydantic;
2. alterar a instrução;
3. testar pelo menos três mensagens;
4. explicar um erro cometido pelo modelo;
5. mostrar como o Pydantic ou uma regra Python ajuda a controlar esse erro.


In [ ]:
# ESPAÇO PARA A NOVA VERSÃO DO GRUPO

# 1. Crie uma nova classe, por exemplo: QualificacaoClienteV2
# 2. Crie uma nova instrução.
# 3. Crie uma nova função de interpretação.
# 4. Execute os testes.


# Desafio opcional — Tentativa de correção automática

Quando a resposta vier inválida, podemos enviar ao modelo:

- a resposta anterior;
- o erro de validação;
- a instrução para corrigir apenas o JSON.

Esse mecanismo será estudado com mais profundidade quando trabalharmos o ciclo de execução do agente.

Pseudocódigo:

```python
resposta = gerar()

try:
    validar(resposta)
except Erro:
    nova_resposta = gerar(
        "Corrija o JSON anterior conforme este erro..."
    )
```


# O que construímos

Ao final desta aula, o protótipo já consegue:

```text
Mensagem em português
        ↓
Modelo aberto executado no Colab
        ↓
Extração de informações
        ↓
JSON
        ↓
Validação com Pydantic
        ↓
Regras Python
        ↓
Perguntar, consultar ou encaminhar
```

## O que ainda não construímos

O protótipo ainda não executa ferramentas automaticamente.

Na próxima aula, acrescentaremos funções como:

- consultar uma base fictícia de imóveis;
- calcular valores;
- devolver o resultado da ferramenta ao modelo;
- controlar o número máximo de etapas do agente.


# Checklist de entrega

A dupla ou trio deverá apresentar:

- [ ] modelo aberto carregado sem API comercial;
- [ ] pelo menos uma mensagem interpretada;
- [ ] resposta em JSON;
- [ ] validação com Pydantic;
- [ ] regra Python para escolher a próxima ação;
- [ ] cinco cenários de teste;
- [ ] pelo menos um erro identificado;
- [ ] uma melhoria realizada na instrução ou no modelo de dados;
- [ ] explicação da diferença entre modelo e agente.

## Conclusão

O protótipo não é apenas um conjunto de respostas prontas: um modelo de linguagem aberto está interpretando mensagens reais. Entretanto, a IA não deve controlar tudo. A combinação entre modelo, validação, regras e supervisão humana torna a solução mais previsível e segura.
